# 07 — RAG eval (retrieval vrstva)

End-to-end eval (notebook `03_evals`) ti při failu **nepoví, jestli selhalo retrieval nebo generation.** RAG eval rozkládá agenta na vrstvy a měří **jen retrieval** — `search_products` voláme přímo, bez LLM kolem.

**Tři failure módy retrievalu:**
1. **Missing retrieval** — relevantní produkt vůbec není v top-k
2. **Špatné pořadí** — relevantní zásah je až na pozici 5, model si ho nevšimne
3. **Šum v top-k** — irelevantní produkty kolem správného zhoršují grounding

**Tři metriky, každá pro jiný failure mód:**

| Metrika | Typ | Co měří |
|---|---|---|
| `recall_at_k` | code | Kolik z relevantních produktů je v top-k? |
| `mrr` | code | Kde v ranku je první relevantní zásah? (1.0 = první pozice) |
| `context_relevance` | LLM judge | Kolik z retrieved produktů je topicky relevantní? |

In [ ]:
from kosik_workshop.config import load_env
load_env()

## 1. Seed retrieval datasetu

Druhý dataset vedle `kosik-eval-golden` — `kosik-retrieval-golden` má jiné schéma:
- `inputs`: `query`, `filters` (passthrough do `search_products`), `k`
- `outputs`: `relevant_ids` (ground truth slugy z katalogu)

Schema enforcement je stejný jako u E2E datasetu — UI „Add to Dataset" nesedící rows odmítne.

In [ ]:
from kosik_workshop.evals import (
    seed_retrieval_dataset,
    RETRIEVAL_EXAMPLES,
    RETRIEVAL_DATASET_NAME,
)

dataset_id = seed_retrieval_dataset()
print(f'Dataset: {RETRIEVAL_DATASET_NAME} ({dataset_id})')
print(f'Examples in code: {len(RETRIEVAL_EXAMPLES)}')

# Ukázka prvního příkladu
RETRIEVAL_EXAMPLES[0]

## 2. Evaluators

Všechny tři vrátí skóre 0–1 per example. Recall a MRR jsou deterministické (zdarma), `context_relevance` je LLM judge (cca $0.001 per example při gpt-4o-mini).

In [ ]:
from kosik_workshop.evals import recall_at_k, mrr, context_relevance, RETRIEVAL_EVALUATORS
[e.__name__ for e in RETRIEVAL_EVALUATORS]

## 3. Spuštění retrieval eval

`build_retrieval_target()` volá `search_products.invoke(...)` přímo — **bez agenta, bez LLM v retrievalu.** Vrací `retrieved_ids` (pro recall/MRR) a `retrieved` (pro context_relevance judge).

`max_concurrency=1` — in-process Chroma client není thread-safe. Po přechodu na server-backed vector store lze zvýšit.

In [ ]:
from kosik_workshop.evals import run_retrieval_evaluation

results = run_retrieval_evaluation(experiment_prefix='kosik-retrieval-baseline')
results

In [ ]:
# Souhrn skóre per evaluator
import pandas as pd

df = results.to_pandas()
score_cols = [c for c in df.columns if c.startswith('feedback.')]
df[score_cols].mean().sort_values()

In [ ]:
# Failures: které dotazy mají recall < 1 (chybí relevantní produkt v top-k)
low_recall = df[df['feedback.recall_at_k'] < 1]
for _, row in low_recall.iterrows():
    q = row.get('inputs.query', '?')
    expected = row.get('reference.relevant_ids', [])
    retrieved = row.get('outputs.retrieved_ids', [])
    print(f'❌ {q!r}')
    print(f'   expected: {list(expected)}')
    print(f'   retrieved: {list(retrieved)}\n')

## 4. A-ha moment: rozbijeme retrieval

Záměrně zhoršíme search query (přidáme šum) → recall a MRR spadnou, ale **E2E eval z notebooku 03 nemusí**, protože LLM si občas poradí i s horším retrievalem. Tohle je přesně důvod, proč evaluovat retrieval samostatně.

In [ ]:
# Variant: query rewriter, který přidá šum (simulujeme regresi v search pipeline)
from kosik_workshop.evals.runner import build_retrieval_target
from langsmith.evaluation import evaluate
from kosik_workshop.tools import search_products

def noisy_target(inputs):
    # Záměrně přidáme generický šum, který rozhodí embedding similarity
    noisy_query = f'{inputs["query"]} levné akce sleva nabídka týden'
    kwargs = {'query': noisy_query}
    kwargs.update(inputs.get('filters') or {})
    if 'k' in inputs:
        kwargs['k'] = inputs['k']
    results = search_products.invoke(kwargs)
    return {
        'retrieved_ids': [r['id'] for r in results],
        'retrieved': results,
    }

noisy_results = evaluate(
    noisy_target,
    data=RETRIEVAL_DATASET_NAME,
    evaluators=RETRIEVAL_EVALUATORS,
    experiment_prefix='kosik-retrieval-noisy',
    max_concurrency=1,
    metadata={'variant': 'noisy_query'},
)
noisy_results

In [ ]:
# Porovnání baseline vs noisy — průměrná skóre
noisy_df = noisy_results.to_pandas()
comparison = pd.DataFrame({
    'baseline': df[score_cols].mean(),
    'noisy': noisy_df[score_cols].mean(),
})
comparison['delta'] = comparison['noisy'] - comparison['baseline']
comparison

Otevři v LangSmith UI: **Datasets → `kosik-retrieval-golden` → Experiments → Compare** dvou runs (`baseline` a `noisy`). Per-example diff ukáže přesně, kde se retrieval rozbil.

## 5. Co dál

- **Eval gate v CI:** `scripts/ci_eval_gate.py` pouští E2E i retrieval eval na každý PR. Threshold pro retrieval default `0.5` (LLM judge je striktní), pro E2E `0.8`.
- **Když retrieval propadne:** zkus jiný embedding model, re-ranker (Cohere Rerank, BGE), nebo hybrid BM25 + vector. Pak re-run tento notebook → měřítelný signál, jestli to pomohlo.
- **Když recall je OK ale `context_relevance` propadá:** retrieval vrací správné produkty, ale s šumem → menší `k`, lepší metadata filtering.
- **Když `context_relevance` je vysoký ale E2E padá:** retrieval je dobrý, problém je v promptu/generaci → zpátky do `03_evals` a A/B prompty (`05_ab_prompts`).